In [10]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path

# Paths to the raw datasets relative to src/model/
DATASET_DIR = Path("../../Dataset").resolve()
# NOTE: Using exactly the two separate NetCDF files present in the Dataset folder
merra_path = DATASET_DIR / "MERRA2_5Day_Averages.nc"
sentinel_path = DATASET_DIR / "S5PL2_5D.nc"

print("Loading raw datasets...")
merra = xr.open_dataset(merra_path)
sentinel = xr.open_dataset(sentinel_path)

def print_dataset_info(name, ds):
    print(f"--- {name} ---")
    if 'time' in ds.coords:
        print(f"Time steps: {len(ds.time)} (From {ds.time.values[0]} to {ds.time.values[-1]})")
    if 'lat' in ds.coords and 'lon' in ds.coords:
        print(f"Spatial grid: {len(ds.lat)} latitudes x {len(ds.lon)} longitudes")
        print(f"Grid bounds: Lat [{ds.lat.min().item():.2f}, {ds.lat.max().item():.2f}], Lon [{ds.lon.min().item():.2f}, {ds.lon.max().item():.2f}]")
    print()

print_dataset_info("MERRA-2 (5-Day Averages)", merra)
print_dataset_info("Sentinel-5P (5-Day Averages)", sentinel)

Loading raw datasets...
--- MERRA-2 (5-Day Averages) ---
Time steps: 506 (From 2019-01-01T00:00:00.000000000 to 2025-11-30T00:00:00.000000000)
Spatial grid: 38 latitudes x 53 longitudes
Grid bounds: Lat [25.00, 34.25], Lon [68.44, 84.69]

--- Sentinel-5P (5-Day Averages) ---
Time steps: 366 (From 2019-01-03T12:00:00.000000000 to 2024-01-02T12:00:00.000000000)
Spatial grid: 291 latitudes x 512 longitudes
Grid bounds: Lat [24.90, 34.36], Lon [68.15, 84.82]



In [11]:
# Step 1: Temporal Alignment
# Sentinel's 5-day averages are stamped 2.5 days after MERRA's (e.g., Jan 3 vs Jan 1)
# To fix the zero overlap, we can use xarray's powerful 'nearest' reindexing 
# to force Sentinel to adopt MERRA-2's exact timeline.

end_date = "2023-12-31"

print(f"Limiting MERRA-2 timeline up to {end_date}...")
merra_time = merra.sel(time=slice(None, end_date))

# Align the Sentinel timeline onto the exact timestep array defined by MERRA-2.
# This prevents 0-overlap errors caused by slightly offset interval centers (Jan 1 vs Jan 3).
print("Aligning Sentinel-5P timeline to MERRA-2 points using nearest neighbor...")
sentinel_aligned_time = sentinel.reindex(time=merra_time.time, method='nearest')
merra_aligned = merra_time

print(f"Temporally aligned both datasets to {len(merra_aligned.time)} overlapping timesteps.")
print(f"New Time boundaries: {merra_aligned.time.min().item()} ===> {merra_aligned.time.max().item()}\n")

Limiting MERRA-2 timeline up to 2023-12-31...
Aligning Sentinel-5P timeline to MERRA-2 points using nearest neighbor...
Temporally aligned both datasets to 366 overlapping timesteps.
New Time boundaries: 1546300800000000000 ===> 1703980800000000000



In [12]:
# Step 2 & 3: Spatial grids alignment via downscaling
# Sentinel-5P generally has far higher resolution than MERRA-2.
# We will use xarray's `.interp()` functionality to automatically downscale 
# (and grid-align) Sentinel's values onto MERRA's lower-resolution lat/lon coordinates.

print("Downscaling and aligning Sentinel-5P spatial grid onto MERRA-2's coordinates...")

# Use nearest-neighbor interpolation to cast S5P lat/lons exactly to MERRA-2
# This handles the spatial alignment automatically
sentinel_aligned = sentinel_aligned_time.interp(
    lat=merra_aligned.lat, 
    lon=merra_aligned.lon, 
    method="nearest"
)

print("-" * 50)
print("FINAL PREPROCESSED DATASET DIMENSIONS:")
print(f"MERRA-2:     Lat/Lon = {len(merra_aligned.lat)} x {len(merra_aligned.lon)}, Time = {len(merra_aligned.time)}")
print(f"Sentinel-5P: Lat/Lon = {len(sentinel_aligned.lat)} x {len(sentinel_aligned.lon)}, Time = {len(sentinel_aligned.time)}")
print("-" * 50)

# Create a subdirectory to save results
output_dir = DATASET_DIR / "preprocessed"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\nCreated output directory at: {output_dir}")

merra_aligned.to_netcdf(output_dir / "merra_5d_preprocessed.nc")
sentinel_aligned.to_netcdf(output_dir / "sentinel_5d_preprocessed.nc")
print("Saved final preprocessed versions successfully!")

Downscaling and aligning Sentinel-5P spatial grid onto MERRA-2's coordinates...
--------------------------------------------------
FINAL PREPROCESSED DATASET DIMENSIONS:
MERRA-2:     Lat/Lon = 38 x 53, Time = 366
Sentinel-5P: Lat/Lon = 38 x 53, Time = 366
--------------------------------------------------

Created output directory at: C:\Users\hp\Documents\GitHub\fifth-season\Dataset\preprocessed
Saved final preprocessed versions successfully!
